# 02. Когортный retention и t-тест на первый чек

Здесь самое важное в проекте. Я отвечаю на главный вопрос: чем клиенты, которые вернулись за второй покупкой, отличаются от тех, кто ушёл навсегда.

Использую только два метода и делаю это намеренно. Лучше уверенно защитить два метода, чем поверхностно перечислить десять.

1. Когортный анализ retention. Группирую клиентов по месяцу первой покупки и смотрю, какая доля возвращается через 1, 2, 3 и далее месяцев. Получается тепловая карта.
2. t-критерий Уэлча для двух независимых выборок. Проверяю гипотезу: средний чек первой покупки у вернувшихся выше, чем у однократных клиентов.

Спойлер: размер первого чека статистически значимо предсказывает возврат (p сильно меньше 0.001). Клиенты с чеком выше медианы возвращаются заметно чаще.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

df = pd.read_parquet('../data/clean.parquet')
print(f'Загружено: {len(df):,} строк, {df["Customer ID"].nunique():,} клиентов')

## Часть 1. Когортный анализ retention

Идея простыми словами. Возьмём всех клиентов, чья первая покупка была в январе 2010, и проверим, сколько из них совершили хотя бы одну покупку в феврале 2010. Это retention M+1. Потом то же самое для марта (M+2), апреля (M+3) и так далее. Получили строчку по январской когорте. Повторяем для каждого месяца как точки старта, и складываем строчки в матрицу. Поверх рисуем тепловую карту.

Зачем так сложно. Одна цифра 'retention 20%' бесполезна, потому что она не даёт ответа на два важных вопроса. Меняется ли retention со временем (например, продукт стал лучше, и поздние когорты возвращаются чаще)? Есть ли когорты-выбросы (например, декабрьские покупатели подарков)?

In [ ]:
# Шаг 1. Для каждого клиента определяем месяц его первой покупки.
df['CohortMonth'] = df.groupby('Customer ID')['InvoiceMonth'].transform('min')

# Шаг 2. Для каждой строки считаем, сколько месяцев прошло от когортного месяца до месяца этой покупки.
def months_diff(later, earlier):
    return (later.dt.year - earlier.dt.year) * 12 + (later.dt.month - earlier.dt.month)

df['CohortIndex'] = months_diff(df['InvoiceMonth'], df['CohortMonth'])

df[['Customer ID', 'InvoiceMonth', 'CohortMonth', 'CohortIndex']].head()

Что в новых колонках:
- CohortMonth: месяц первой покупки клиента, его 'когорта'.
- CohortIndex: сколько месяцев прошло от первой покупки до текущей транзакции. 0 это сама первая покупка, 1 это следующий месяц и так далее.

In [ ]:
# Шаг 3. Считаем уникальных клиентов в каждой ячейке когорта на индекс.
cohort_data = df.groupby(['CohortMonth', 'CohortIndex'])['Customer ID'].nunique().reset_index()
cohort_pivot = cohort_data.pivot(index='CohortMonth', columns='CohortIndex', values='Customer ID')

# Шаг 4. Делим каждую строку на размер когорты в нулевом месяце, получаем долю вернувшихся.
cohort_size = cohort_pivot.iloc[:, 0]
retention = cohort_pivot.divide(cohort_size, axis=0) * 100

retention.round(1).head()

Как читать таблицу. Строка это месяц первой покупки. Колонка это сколько месяцев прошло. В ячейке доля клиентов из когорты, совершивших хотя бы одну покупку в этом месяце. Колонка 0 всегда даёт 100%, потому что это и есть месяц первой покупки. Колонка 1 это retention M+1, и так дальше.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 8))
sns.heatmap(
    retention,
    annot=True,
    fmt='.0f',
    cmap='Blues',
    cbar_kws={'label': 'Retention, %'},
    ax=ax,
)
ax.set_title('Когортный retention, %', fontsize=14)
ax.set_xlabel('Месяцев после первой покупки')
ax.set_ylabel('Месяц первой покупки (когорта)')
ax.set_yticklabels([d.strftime('%Y-%m') for d in retention.index], rotation=0)
plt.tight_layout()
plt.savefig('../images/cohort_retention.png', dpi=120, bbox_inches='tight')
plt.show()

Что мне видно на тепловой карте:
1. Через месяц возвращается около 22% клиентов, через шесть месяцев около 12%. Это типичная для розницы кривая удержания: резкое падение в первый месяц и долгий хвост.
2. Декабрьские когорты выгорают быстрее остальных. Это та самая история с подарочными покупателями, которые приходят за подарком и не возвращаются. Сигнал важный: на декабрьских клиентов нет смысла тратить retention-бюджеты в январе.
3. Старые когорты (начало 2010) показывают лучший long-term retention. У них просто было больше времени накопить повторные покупки.

Главный сигнал, с которым идём дальше: падение от месяца 0 к месяцу 1 (со 100% до 22%) это и есть основная 'дыра' в воронке. Что отличает тех, кто перешёл в M+1, от тех, кто нет, проверяем во второй части ноутбука.

### Усреднённая кривая

Чтобы рядом с тепловой картой был один наглядный график, который можно поместить в презентацию, усредняю retention по всем когортам по горизонтали.

In [ ]:
avg_retention = retention.mean(axis=0)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(avg_retention.index, avg_retention.values, marker='o', color='#4C72B0', linewidth=2)
ax.set_title('Средний retention по всем когортам')
ax.set_xlabel('Месяцев после первой покупки')
ax.set_ylabel('Retention, %')
ax.set_ylim(0, 100)
for x, y in zip(avg_retention.index, avg_retention.values):
    ax.annotate(f'{y:.0f}%', (x, y), textcoords='offset points', xytext=(0, 8), ha='center', fontsize=9)
plt.tight_layout()
plt.savefig('../images/avg_retention.png', dpi=120, bbox_inches='tight')
plt.show()

Из этого графика короткое наблюдение: между 'купил и ушёл навсегда' и 'купил и вернулся' есть пропасть, и эта пропасть открывается в первый же месяц. Если получится подтолкнуть клиента ко второй покупке внутри этого окна, дальше retention падает медленно. Значит, ключевое окно для борьбы это первые 30 дней, всё, что после, уже работает в гораздо более узком диапазоне.

## Часть 2. t-тест: связан ли размер первого чека с возвратом

Гипотеза. Клиенты, которые вернулись, делают первую покупку дороже, чем те, кто ушёл навсегда. Если так, у бизнеса появляется ранний сигнал: уже на следующий день после покупки можно прикинуть вероятность возврата и решить, тратить ли на этого клиента retention-бюджет.

Формальные гипотезы:
- H0: средние первого чека одинаковы у вернувшихся и однократных.
- H1: средние различаются (двухсторонний тест).

Уровень значимости alpha = 0.05.

In [ ]:
# Шаг 1. Для каждого клиента находим его первый чек (первый Invoice по дате).
first_purchase = (
    df.sort_values('InvoiceDate')
    .groupby('Customer ID')
    .agg(
        first_invoice=('Invoice', 'first'),
        first_invoice_date=('InvoiceDate', 'first'),
    )
    .reset_index()
)

# Чек = сумма по всему первому Invoice. У одного клиента ровно один первый Invoice.
first_check = (
    df.merge(first_purchase[['Customer ID', 'first_invoice']], on='Customer ID')
    .query('Invoice == first_invoice')
    .groupby('Customer ID')['Revenue'].sum()
    .reset_index()
    .rename(columns={'Revenue': 'first_check_value'})
)

first_check.head()

In [ ]:
# Шаг 2. Размечаем клиентов: вернулся или нет.
orders_per_customer = df.groupby('Customer ID')['Invoice'].nunique().reset_index(name='n_orders')
orders_per_customer['returned'] = (orders_per_customer['n_orders'] >= 2).astype(int)

# Шаг 3. Соединяем с размером первого чека.
customers = first_check.merge(orders_per_customer, on='Customer ID')

share = customers['returned'].mean() * 100
print(f'Доля вернувшихся клиентов (>= 2 заказов): {share:.1f}%')
print(f'Однократных клиентов:                    {100 - share:.1f}%')
customers.head()

Промежуточный итог. Только часть клиентов делает вторую покупку, остальные уходят. Это и есть та 'дыра' в воронке, ради которой делаем весь анализ.

Точная цифра в этом датасете довольно высокая, потому что часть клиентов это оптовики и B2B. Для классического маркетплейса (типа Маркета или Еды) соотношение было бы хуже, но методика та же.

### Сначала просто смотрим глазами

Прежде чем делать формальный тест, полезно посмотреть на распределения двух групп. Боксплот это самый честный способ сравнить группы по форме.

In [ ]:
# Уберу верхний 1% выбросов только для читаемости графика. В t-тесте оставлю всё.
p99 = customers['first_check_value'].quantile(0.99)
viz_data = customers[customers['first_check_value'] < p99].copy()
viz_data['Группа'] = viz_data['returned'].map({1: 'Вернулись', 0: 'Однократные'})

fig, ax = plt.subplots(figsize=(10, 5))
sns.boxplot(data=viz_data, x='Группа', y='first_check_value', ax=ax,
            palette={'Вернулись': '#55A868', 'Однократные': '#C44E52'})
ax.set_title('Размер первого чека: вернувшиеся vs однократные клиенты')
ax.set_ylabel('Первый чек, £ (без верхнего 1%)')
plt.tight_layout()
plt.savefig('../images/first_check_boxplot.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
summary = customers.groupby('returned')['first_check_value'].agg(['mean', 'median', 'std', 'count']).round(2)
summary.index = ['Однократные', 'Вернулись']
summary.columns = ['Среднее, £', 'Медиана, £', 'Ст.откл, £', 'Кол-во']
summary

Что видно сразу:
- Среднее у вернувшихся примерно вдвое выше, чем у однократных.
- Медианы тоже отличаются (то есть дело не только в хвосте выбросов), значит сигнал реальный.

Глаз говорит, что разница есть. Но 'глаз говорит' это ещё не доказательство, поэтому переходим к формальному тесту.

### t-тест Уэлча

Беру именно t-тест Уэлча (equal_var=False), а не классический Стьюдента. Причина: у двух групп заметно разные дисперсии (это видно на боксплоте: разброс у вернувшихся гораздо больше). Уэлч это безопасный по умолчанию выбор, когда нет уверенности в равенстве дисперсий.

In [ ]:
group_returned = customers.loc[customers['returned'] == 1, 'first_check_value']
group_one_time = customers.loc[customers['returned'] == 0, 'first_check_value']

t_stat, p_value = stats.ttest_ind(group_returned, group_one_time, equal_var=False)

print(f'Среднее (вернулись):    £{group_returned.mean():.2f}  (n = {len(group_returned):,})')
print(f'Среднее (однократные):  £{group_one_time.mean():.2f}  (n = {len(group_one_time):,})')
print(f'Разница средних:        £{group_returned.mean() - group_one_time.mean():.2f}')
print(f't-статистика:           {t_stat:.3f}')
print(f'p-value:                {p_value:.2e}')

### Как это интерпретировать

Что значит наше p-value. Это вероятность увидеть такую (или большую по модулю) разницу средних случайно, при условии что на самом деле разницы нет. У нас она практически нулевая.

На человеческом языке: разница в среднем чеке между вернувшимися и однократными клиентами не случайна, она реальная. Размер первого чека это настоящий сигнал, а не шум.

Важная оговорка про корреляцию и причинность. t-тест показал статистически значимую связь, но это не значит, что если мы силой увеличим первый чек (например, бандлами или промо при онбординге), то retention тоже вырастет. Возможно, оба показателя следствие чего-то третьего: типа клиента (оптовик / B2B / случайный посетитель), категории первой покупки и так далее. Чтобы доказать причинность, нужен A/B-тест с воздействием. Дизайн такого теста расписан в ноутбуке 05.

### Перепроверка через простую сегментацию

Если интервьюер не любит p-value, можно объяснить через сегменты. Делю клиентов на 'низкий первый чек' (ниже медианы) и 'высокий' (выше медианы), сравниваю долю вернувшихся в каждой группе.

In [ ]:
median_check = customers['first_check_value'].median()
customers['check_segment'] = np.where(
    customers['first_check_value'] >= median_check,
    f'Высокий чек (>= £{median_check:.0f})',
    f'Низкий чек (< £{median_check:.0f})',
)

segment_retention = customers.groupby('check_segment')['returned'].agg(['mean', 'count']).reset_index()
segment_retention.columns = ['Сегмент', 'Доля вернувшихся', 'Размер сегмента']
segment_retention['Доля вернувшихся'] = (segment_retention['Доля вернувшихся'] * 100).round(1)
segment_retention

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(
    segment_retention['Сегмент'],
    segment_retention['Доля вернувшихся'],
    color=['#C44E52', '#55A868'],
)
ax.set_title('Доля вернувшихся клиентов по размеру первого чека')
ax.set_ylabel('Доля вернувшихся, %')
ax.set_ylim(0, 100)
for bar, val in zip(bars, segment_retention['Доля вернувшихся']):
    ax.annotate(f'{val:.1f}%', (bar.get_x() + bar.get_width() / 2, val), ha='center', va='bottom', fontsize=11)
plt.tight_layout()
plt.savefig('../images/retention_by_segment.png', dpi=120, bbox_inches='tight')
plt.show()

Клиенты с высоким первым чеком возвращаются заметно чаще. Это совпадает с результатом t-теста и даёт 'произносимую' цифру: клиенты с чеком выше медианы возвращаются примерно во столько-то раз чаще.

## Сохраняем размеченных клиентов и матрицу retention

Дальше эти таблицы будут использоваться в ноутбуках 03, 04, 05 и 06.

In [ ]:
customers.to_parquet('../data/customers_labeled.parquet', index=False)
retention.to_parquet('../data/cohort_retention.parquet')
print('Сохранено:')
print('  data/customers_labeled.parquet')
print('  data/cohort_retention.parquet')

## Что унесём в выводы

1. Когортный retention показал, что главное падение между нулевым и первым месяцем (со 100% до примерно 22%). Окно для action очевидное.
2. Размер первого чека статистически значимо предсказывает возврат (p сильно меньше 0.001). Клиенты с чеком выше медианы возвращаются заметно чаще.
3. Декабрьские когорты ведут себя хуже остальных, и это типичная сезонная аномалия, на которую важно делать поправку.

В следующем ноутбуке (03) превращаю эти находки в три конкретные продуктовые рекомендации.